# The Salary Is Right

## An Autonomous Agentic Framework for Salary Opportunities

This notebook adapts the Week 8 "The Price Is Right" product pricing system to **salary prediction** for the job market.

**Architecture:**
- **Scanner Agent** — scrapes job listing RSS feeds, uses structured outputs to parse them
- **Specialist Agent** — calls our fine-tuned salary model deployed on Modal.com
- **Frontier Agent** — RAG pipeline: finds similar jobs in ChromaDB, asks GPT to estimate salary
- **Ensemble Agent** — combines specialist + frontier estimates
- **Messaging Agent** — sends push notifications for above-market opportunities
- **Planning Agent** — orchestrates everything with an agent loop + tool calling

**Key adaptations from the course:**
- Products → Job listings
- Product prices → Annual salaries
- Deal discount → Salary premium (offered vs. market rate)
- DealNews RSS → RemoteOK / WeWorkRemotely RSS

---
## Part 1: Setup

In [ ]:
import os
import sys
import logging
from dotenv import load_dotenv

load_dotenv(override=True)

logging.basicConfig(level=logging.INFO, format='[%(asctime)s] %(message)s', datefmt='%H:%M:%S')

print("OPENAI_API_KEY found:", "OPENAI_API_KEY" in os.environ)
print("HF_TOKEN found:", "HF_TOKEN" in os.environ)
print("PUSHOVER_USER found:", "PUSHOVER_USER" in os.environ)

---
## Part 2: Scanner Agent — Structured Outputs

The Scanner Agent scrapes job listing RSS feeds, then uses OpenAI structured outputs to parse
unstructured HTML into clean `JobListing` Pydantic objects.

We start with test data to verify everything works without needing live RSS feeds.

In [ ]:
from agents.scanner_agent import ScannerAgent

scanner = ScannerAgent()

In [ ]:
# Use test data first — no API calls needed
test_results = scanner.test_scan()

print(f"Got {len(test_results.listings)} test listings:\n")
for listing in test_results.listings:
    print(f"  {listing.job_title} at {listing.company}")
    print(f"  Location: {listing.location} | Salary: ${listing.salary:,.0f}")
    print(f"  {listing.description}")
    print()

In [ ]:
# Let's see the Pydantic model's JSON schema — this is what gets sent to OpenAI
# for structured output constrained decoding
from agents.job_listings import JobSelection
import json

print(json.dumps(JobSelection.model_json_schema(), indent=2))

### (Optional) Live scan from RSS feeds
Uncomment the cell below to scan real job listing RSS feeds. This makes HTTP requests to RemoteOK and WeWorkRemotely, then calls OpenAI with structured outputs.

In [ ]:
# Uncomment to try live scanning:

live_results = scanner.scan(memory=[])
if live_results:
    for listing in live_results.listings:
        print(f"{listing.job_title} at {listing.company}: ${listing.salary:,.0f}")
        print(f"  {listing.description}\n")
else:
    print("No listings found — RSS feeds may be unavailable")

---
## Part 3: Populate ChromaDB with Job Data

For the RAG pipeline, we need a vector store of jobs with known salaries.
We load the `lukebarousse/data_jobs` dataset, encode each job with
`all-MiniLM-L6-v2`, and store in ChromaDB.

**This only needs to run once** — the data persists in `jobs_vectorstore/`.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from tqdm import tqdm

DB_PATH = "jobs_vectorstore"

client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_or_create_collection("jobs")
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print(f"ChromaDB collection currently has {collection.count()} items")

In [ ]:
# Set to True to use the full dataset, False for a faster ~20k subset
LITE_MODE = True

dataset = load_dataset("lukebarousse/data_jobs", split="train")
print(f"Loaded {len(dataset)} rows from the dataset")
print(f"Columns: {dataset.column_names}")

In [ ]:
# Preview a sample row to see the field names
sample = dataset[0]
for key, value in sample.items():
    val_str = str(value)[:100] if value else "None"
    print(f"  {key}: {val_str}")

In [ ]:
MIN_SALARY = 15_000
MAX_SALARY = 500_000

if collection.count() > 0:
    print(f"ChromaDB already populated with {collection.count()} jobs — skipping")
    print("Delete the jobs_vectorstore/ folder and re-run to rebuild")
else:
    # Filter to jobs with valid salaries
    valid_rows = []
    for row in dataset:
        salary = row.get("salary_year_avg")
        if salary is not None and salary == salary:  # NaN check
            salary = float(salary)
            if MIN_SALARY <= salary <= MAX_SALARY:
                title = row.get("job_title") or row.get("job_title_short") or ""
                if title:
                    valid_rows.append((row, salary))

    if LITE_MODE:
        valid_rows = valid_rows[:20_000]

    print(f"Processing {len(valid_rows)} jobs with valid salaries...")

    batch_size = 500
    for i in tqdm(range(0, len(valid_rows), batch_size)):
        batch = valid_rows[i:i + batch_size]
        texts = []
        salaries = []
        ids = []

        for j, (row, salary) in enumerate(batch):
            title = row.get("job_title") or row.get("job_title_short") or ""
            company = row.get("company_name") or ""
            location = row.get("job_location") or ""
            skills = row.get("job_skills") or ""
            schedule = row.get("job_schedule_type") or ""

            text = f"Job Title: {title}"
            if company:
                text += f"\nCompany: {company}"
            if location:
                text += f"\nLocation: {location}"
            if schedule:
                text += f"\nType: {schedule}"
            if skills:
                text += f"\nSkills: {skills}"

            texts.append(text[:500])
            salaries.append(salary)
            ids.append(str(i + j))

        vectors = encoder.encode(texts).tolist()
        metadatas = [{"salary": s} for s in salaries]
        collection.add(
            documents=texts,
            embeddings=vectors,
            ids=ids,
            metadatas=metadatas,
        )

    print(f"\nDone! ChromaDB now has {collection.count()} jobs")

### Visualize the vector space (optional)
Use t-SNE to project 384-dimensional job vectors down to 2D.

In [ ]:
import numpy as np
from sklearn.manifold import TSNE
import plotly.express as px

sample_size = min(2000, collection.count())
result = collection.get(include=["embeddings", "documents", "metadatas"], limit=sample_size)
vectors = np.array(result["embeddings"])
salaries_viz = [m["salary"] for m in result["metadatas"]]

# Bin salaries into ranges for coloring
salary_bins = []
for s in salaries_viz:
    if s < 60_000:
        salary_bins.append("< $60k")
    elif s < 100_000:
        salary_bins.append("$60k-$100k")
    elif s < 150_000:
        salary_bins.append("$100k-$150k")
    elif s < 200_000:
        salary_bins.append("$150k-$200k")
    else:
        salary_bins.append("$200k+")

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
reduced = tsne.fit_transform(vectors)

fig = px.scatter(
    x=reduced[:, 0], y=reduced[:, 1], color=salary_bins,
    title=f"Job Vectors in 2D (t-SNE) — {sample_size} jobs",
    labels={"x": "t-SNE 1", "y": "t-SNE 2", "color": "Salary Range"},
    width=900, height=600,
)
fig.update_traces(marker=dict(size=3))
fig.show()

### Test RAG lookup
Find 5 similar jobs to a given description.

In [ ]:
test_query = "Senior Data Scientist with Python and machine learning experience"
vector = encoder.encode([test_query])
results = collection.query(query_embeddings=vector.astype(float).tolist(), n_results=5)

print(f"Similar jobs to: '{test_query}'\n")
for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(f"  Salary: ${meta['salary']:,.0f}")
    print(f"  {doc[:150]}...\n")

---
## Part 4: Frontier Agent — RAG + GPT

The Frontier Agent finds 5 similar jobs in ChromaDB, includes their salaries
as context, and asks GPT to estimate the salary. This is inference-time
techniques (RAG) vs. the training-time techniques (fine-tuning) of the Specialist.

In [ ]:
from agents.frontier_agent import FrontierAgent

frontier = FrontierAgent(collection)

In [ ]:
test_jobs = [
    "Senior Data Scientist at a tech company in San Francisco. Requires 5+ years Python, ML, deep learning.",
    "Junior Software Developer, entry-level React and Node.js. Remote position.",
    "Machine Learning Engineer building production NLP pipelines. PyTorch, transformers, AWS.",
]

for job in test_jobs:
    estimate = frontier.estimate(job)
    print(f"  '{job[:60]}...' → ${estimate:,.0f}\n")

---
## Part 5: Specialist Agent — Fine-Tuned Model on Modal

The Specialist Agent calls our fine-tuned Llama 3.2 3B model running on Modal.com.

**Prerequisites:**
1. Edit `salary_service.py` with your HF_USER, RUN_NAME, and REVISION from Week 7
2. Run `modal deploy salary_service.py` in your terminal
3. Make sure Modal has your HuggingFace secret: `modal secret create huggingface-secret HF_TOKEN=hf_...`

In [ ]:
from agents.specialist_agent import SpecialistAgent

# This will connect to Modal — make sure you've deployed first!
specialist = SpecialistAgent()

In [ ]:
# First call takes ~30s (cold start), subsequent calls are fast
for job in test_jobs:
    estimate = specialist.estimate(job)
    print(f"  '{job[:60]}...' → ${estimate:,.0f}\n")

---
## Part 6: Ensemble Agent

Combines Frontier (60%) and Specialist (40%) estimates. The weights are rough —
you could improve by running linear regression on validation data to find optimal weights.

In [ ]:
from agents.ensemble_agent import EnsembleAgent

ensemble = EnsembleAgent(collection)

In [ ]:
for job in test_jobs:
    estimate = ensemble.estimate(job)
    print(f"  '{job[:60]}...' → ${estimate:,.0f}\n")

---
## Part 7: The Agent Loop — Autonomous Planning Agent

This is the heart of the agentic framework. The Planning Agent is an LLM
(GPT-4o) equipped with three tools:
1. `scan_for_job_listings` — calls the Scanner Agent
2. `estimate_market_salary` — calls the Ensemble Agent
3. `notify_user_of_opportunity` — calls the Messaging Agent

The LLM autonomously decides which tools to call in a `while not done` loop.

### First: fake functions to prove the loop works

In [ ]:
from openai import OpenAI
import json

openai_client = OpenAI()

# Test data from the scanner
test_results = scanner.test_scan()

In [ ]:
# Three FAKE functions — they just print and return dummy data

def scan_for_job_listings() -> str:
    print("[FAKE] Scanning for job listings...")
    return test_results.model_dump_json()

def estimate_market_salary(description: str) -> str:
    print(f"[FAKE] Estimating salary for: {description[:50]}...")
    return f"The estimated market salary for this role is $100000"

def notify_user_of_opportunity(description: str, offered_salary: float, estimated_salary: float, url: str) -> str:
    print(f"[FAKE] Notifying user: {description[:50]}... offered=${offered_salary:,.0f}, market=${estimated_salary:,.0f}")
    return "Notification sent"

In [ ]:
# Tool definitions — the JSON specs that describe our tools to the LLM

tools = [
    {"type": "function", "function": {
        "name": "scan_for_job_listings",
        "description": "Scans job listing feeds and returns promising positions with salary info",
        "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False},
    }},
    {"type": "function", "function": {
        "name": "estimate_market_salary",
        "description": "Given a job description, estimate the typical market salary for that role",
        "parameters": {"type": "object", "properties": {
            "description": {"type": "string", "description": "The job description"},
        }, "required": ["description"], "additionalProperties": False},
    }},
    {"type": "function", "function": {
        "name": "notify_user_of_opportunity",
        "description": "Notify the user about the single best salary opportunity; only call once",
        "parameters": {"type": "object", "properties": {
            "description": {"type": "string", "description": "The job description"},
            "offered_salary": {"type": "number", "description": "The salary offered"},
            "estimated_salary": {"type": "number", "description": "The estimated market salary"},
            "url": {"type": "string", "description": "The job listing URL"},
        }, "required": ["description", "offered_salary", "estimated_salary", "url"], "additionalProperties": False},
    }},
]

In [ ]:
# The tool call handler

tool_mapping = {
    "scan_for_job_listings": scan_for_job_listings,
    "estimate_market_salary": estimate_market_salary,
    "notify_user_of_opportunity": notify_user_of_opportunity,
}

def handle_tool_call(message):
    results = []
    for tool_call in message.tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        tool = tool_mapping.get(name)
        result = tool(**args) if tool else ""
        results.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
    return results

In [ ]:
# THE AGENT LOOP — this is the core of agentic AI

messages = [
    {"role": "system", "content": "You find promising job opportunities by scanning listings and estimating whether the offered salary is above market rate, then notify the user of the best one."},
    {"role": "user", "content": "First, scan for job listings. Then estimate the market salary for each. Then pick the single best opportunity where the offered salary is much higher than the market estimate, and notify the user. Reply OK when done."},
]

done = False
while not done:
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, tools=tools
    )
    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        results = handle_tool_call(message)
        messages.append(message)
        messages.extend(results)
    else:
        done = True

print(f"\nAgent completed with: {response.choices[0].message.content}")

The agent autonomously decided the order and number of tool calls!
Even though these were fake functions, the LLM drove the workflow.

### Now: the real Autonomous Planning Agent

Same loop, but the tools now call real agents (Scanner, Ensemble, Messaging).

In [ ]:
from agents.autonomous_planning_agent import AutonomousPlanningAgent

agent = AutonomousPlanningAgent(collection)

In [ ]:
# Run the full autonomous agent loop
# This will: scan RSS feeds → estimate salaries (Modal + RAG) → notify you

opportunity = agent.plan(memory=[])

if opportunity:
    print(f"\n{'='*60}")
    print(f"Best opportunity found!")
    print(f"  Job: {opportunity.listing.job_title}")
    print(f"  Offered: ${opportunity.listing.salary:,.0f}")
    print(f"  Market estimate: ${opportunity.estimate:,.0f}")
    print(f"  Premium: ${opportunity.premium:,.0f}")
    print(f"{'='*60}")
else:
    print("No above-market opportunities found this run.")

---
## Part 8: (Optional) Launch the Gradio UI

The full Gradio interface runs as a persistent app with a 5-minute auto-refresh timer.
Run `python the_salary_is_right.py` from the terminal for the best experience,
or uncomment below to launch from the notebook.

In [ ]:
# Uncomment to launch the Gradio UI from this notebook:

from the_salary_is_right import App
App().run()

---
## Architecture Summary

What just happened across a single run:

| Agent | What it does | Model(s) used |
|-------|-------------|---------------|
| **Scanner** | Scrapes RSS feeds, structured outputs | GPT-4o-mini |
| **Preprocessor** | Rewrites job text into structured format | GPT-4o-mini |
| **Specialist** | Fine-tuned salary prediction | Llama 3.2 3B (on Modal) |
| **Frontier** | RAG lookup + GPT estimation | MiniLM encoder + GPT-4o |
| **Ensemble** | Weighted combination (60/40) | — |
| **Messaging** | Crafts alert + push notification | GPT-4o-mini |
| **Planning** | Agent loop with tool calling | GPT-4o |

**Techniques combined:**
- Fine-tuning (training-time) via the Specialist Agent
- RAG (inference-time) via the Frontier Agent
- Structured outputs for parsing unstructured data
- Tool calling for autonomous agent orchestration
- Ensemble model combining multiple estimation approaches
- Deployment to serverless GPU (Modal.com)